In [ ]:
from functools import partial
from pathlib import Path

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ─────────────────────────────────────────────────────────────────────────────

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# update weather flag
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"

### Activity 4: DES and TEN

In [ ]:
# This is the same as above, but in a new directory
activity_4_analysis_dir = analysis_dir / "activity_4"
if not activity_4_analysis_dir.exists():
    activity_4_analysis_dir.mkdir()

uo_coincident = UO(
    activity_4_analysis_dir, "coincident", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

# run sims faster
uo_coincident.set_number_parallel(8)

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )
    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')

# Create the scenarios
uo_coincident.create_scenarios("class_project_coincident.json")

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name.
uo_coincident.fix_dependencies_20260420("base_workflow.osw")

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# post process/visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

# save these as the relative paths because they will be run from within docker

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Now create the 4G system
uo_coincident.des_params(scenario_path, feature_path, sys_param_path)
uo_coincident.des_create(sys_param_path, feature_path, "four_g")

In [ ]:
# Run the 4G system (run with the modelica runner for now). Even
# though the URBANopt-DES package has a run command

# Run the ORIGINAL 4G Model from uo_cli
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner

mr = ModelicaRunner()

des_scenario_name = "four_g"
des_analysis_4g_dir = activity_4_analysis_dir / des_scenario_name

# print out the full arguments so that we know what was set
print(f"{des_scenario_name}.Districts.district")
print(f"file_to_load={des_analysis_4g_dir / 'package.mo'}")
print(f"run_path={des_analysis_4g_dir}")

print(f"Running model in the following directory: {des_analysis_4g_dir}")

# check if the results exist from previous run and skip if so
results_path = (
    des_analysis_4g_dir / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
)
if (results_path).exists():
    print("Results already exist, skipping model run.")
else:
    # use older version of gmt-om-runner to run the older 4G model
    success, results_path = mr.run_in_docker(
        "compile_and_run",
        f"{des_scenario_name}.Districts.DistrictEnergySystem",
        file_to_load=f"{des_analysis_4g_dir / 'package.mo'}",
        run_path=des_analysis_4g_dir,
        stop_time=3.1536e7,
    )


# Takes about 4 hours to compile and run